# RL-09 : RL offline — Behavior Cloning et erreur d'extrapolation

**Navigation** : [Version Python](RL-09-RL-Offline-Behavior-Cloning-Python.ipynb) · [Plan](#plan)

**Série** : Reinforcement Learning | **Notebook** : 9 | **Durée estimée** : 50-55 min

**Prérequis** : notebook 5 (Q-learning, politiques epsilon-greedy) ; le notebook 8 (Dyna-Q) aide pour le labyrinthe de Sutton & Barto mais n'est pas indispensable.

## Objectifs pedagogiques

- Comprendre la différence entre RL **online** (l'agent interagit) et RL **offline** (un dataset fige de transitions, aucune interaction possible) ;
- Implementer le **Behavior Cloning** (BC) tabulaire et étudier comment ses résultats dépendent de la qualité et du support des démonstrations, y compris le filtrage du bruit ;
- Mettre en evidence l'**erreur d'extrapolation** : pourquoi le Q-learning applique naivement a un dataset fige echoue, même avec des données expertes ;
- Implementer une correction de type **BCQ** (contraindre le bootstrap au support du dataset) et observer le **stitching** : extraire une politique quasi optimale de données mediocres ;
- Relier ces idees aux méthodes modernes (BCQ, CQL) et a l'alignement des LLM (RLHF, DPO).

## Plan

1. Du RL online au RL offline
2. Fabriquer les datasets : trois politiques de comportement
3. Behavior Cloning
4. Q-learning offline naif : l'erreur d'extrapolation
5. Contraindre au support du dataset : BCQ-lite
6. Du tabulaire au deep : BCQ, CQL et le pont vers RLHF/DPO
7. Exercices

## 1. Du RL online au RL offline

Tous les algorithmes vus jusqu'ici (Q-learning, DQN, policy gradient, Dyna-Q) sont **online** : l'agent choisit ses actions, observe les consequences, et corrige. Cette boucle d'essai-erreur est un luxe. En robotique, en sante, en conduite autonome ou en finance, laisser une politique non entrainee explorer librement est couteux, voire dangereux. En revanche, on dispose souvent de **journaux d'interactions passees** : trajectoires d'un opérateur humain, logs d'un ancien controleur, historique de decisions cliniques.

Le **RL offline** (ou *batch RL*) pose la question : peut-on apprendre une bonne politique **uniquement** a partir d'un dataset fige $\mathcal{D} = \{(s_i, a_i, r_i, s'_i)\}$, collecte par une **politique de comportement** $\pi_\beta$ inconnue, sans jamais interagir avec l'environnement pendant l'apprentissage ?

Deux familles de reponses naives, deux echecs instructifs :

- **Imiter** : reproduire l'action majoritaire du dataset dans chaque etat (*Behavior Cloning*). Simple, et ses performances dépendent des données : le filtrage du bruit peut améliorer une politique légèrement bruitée, tandis que BC plafonne souvent quand les données sont médiocres.
- **Faire du Q-learning sur le dataset** : après tout, le Q-learning est *off-policy*, il devrait digerer des données venues d'ailleurs. C'est vrai en apparence seulement : sans nouvelles interactions pour corriger les illusions, le $\max_a Q(s', a)$ bootstrape sur des actions **jamais observees** et l'erreur s'auto-amplifie. C'est l'**erreur d'extrapolation** (Fujimoto et al., 2019), que nous allons reproduire pas a pas.

Nous travaillons en tabulaire sur le labyrinthe 6x9 de Sutton & Barto (fig. 8.2) — le même que le notebook 8 — avec une convention de recompense importante pour la suite : **-1 par pas, 0 en atteignant le but**. Maximiser le retour = minimiser la longueur du chemin, et surtout l'initialisation $Q = 0$ devient **optimiste** : une action jamais essayee semble meilleure que la realite. Retenez ce detail, c'est le carburant de l'erreur d'extrapolation.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Xml.Linq;
using Microsoft.DotNet.Interactive;
using Microsoft.DotNet.Interactive.Formatting;

const int SEED = 42, MAX_STEPS = 200;
const double ALPHA = 0.5, GAMMA = 0.95, EPSILON = 0.1;
var invariant = CultureInfo.InvariantCulture;
string SvgNumber(double value) => value.ToString("0.###", CultureInfo.InvariantCulture);
double F(double x) => double.IsFinite(x) ? x : throw new InvalidOperationException("Valeur non finie");
Console.WriteLine($".NET {Environment.Version} — graine {SEED}; alpha={ALPHA}, gamma={GAMMA}, epsilon={EPSILON}, horizon={MAX_STEPS}");

.NET 10.0.14 — graine 42; alpha=0.5, gamma=0.95, epsilon=0.1, horizon=200


**Lecture.** Le kernel confirme les paramètres de l’expérience (.NET 10.0.14, graine 42). La cellule suivante construit la géométrie du labyrinthe et vérifie l’oracle BFS de 14 pas.

In [2]:
class Maze {
    public const int Height=6, Width=9, NStates=Height*Width, NActions=4;
    public readonly (int r,int c) Start=(2,0), Goal=(0,8);
    public readonly HashSet<(int r,int c)> Obstacles=new(){(1,2),(2,2),(3,2),(4,5),(0,7),(1,7),(2,7)};
    public readonly (int dr,int dc)[] Moves={(-1,0),(1,0),(0,-1),(0,1)};
    public int Id((int r,int c) s)=>s.r*Width+s.c;
    public (int r,int c) Pos(int i)=>(i/Width,i%Width);
    public (int state,double reward,bool done) Step(int s,int a){var p=Pos(s);var m=Moves[a];var q=(r:p.r+m.dr,c:p.c+m.dc);if(q.r<0||q.r>=Height||q.c<0||q.c>=Width||Obstacles.Contains(q))q=p;int j=Id(q);return (j,q==Goal?0.0:-1.0,q==Goal);}
}
var env=new Maze();
int BfsDistance(Maze e){var d=new int[Maze.NStates];Array.Fill(d,-1);var q=new Queue<int>();int start=e.Id(e.Start),goal=e.Id(e.Goal);d[start]=0;q.Enqueue(start);while(q.Count>0){int s=q.Dequeue();if(s==goal)return d[s];for(int a=0;a<4;a++){var t=e.Step(s,a).state;if(d[t]<0){d[t]=d[s]+1;q.Enqueue(t);}}}return -1;}
if(BfsDistance(env)!=14)throw new Exception("Oracle BFS géométrique inattendu");
string mazeSvg=$"<svg xmlns='http://www.w3.org/2000/svg' width='540' height='390' viewBox='0 0 540 390'><rect width='100%' height='100%' fill='white'/><text x='20' y='24' font-size='17'>Labyrinthe 6 × 9 — chemin BFS : {BfsDistance(env)} pas</text>";
for(int r=0;r<6;r++)for(int c=0;c<9;c++){string fill=env.Obstacles.Contains((r,c))?"#555":"#fff";mazeSvg+=$"<rect x='{c*55+20}' y='{r*55+45}' width='54' height='54' fill='{fill}' stroke='#888'/>";if((r,c)==env.Start|| (r,c)==env.Goal)mazeSvg+=$"<text x='{c*55+47}' y='{r*55+80}' text-anchor='middle' font-size='18' font-weight='bold'>{((r,c)==env.Start?"S":"G")}</text>";}
for(int c=0;c<9;c++)mazeSvg+=$"<text x='{c*55+47}' y='382' text-anchor='middle' font-size='11'>{c}</text>";for(int r=0;r<6;r++)mazeSvg+=$"<text x='10' y='{r*55+78}' text-anchor='middle' font-size='11'>{r}</text>";mazeSvg+="</svg>";record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w) => w.Write(v.Content), "text/html");
display(new PlotSvg(mazeSvg));
Console.WriteLine($"Labyrinthe {Maze.Height}×{Maze.Width}; {env.Obstacles.Count} obstacles; {Maze.NStates} cases × 4 actions. Oracle indépendant BFS : 14 pas, retour optimal -13.");

Labyrinthe 6 × 9 — chemin BFS : 14 pas G S 0 1 2 3 4 5 6 7 8 0 1 2 3 4 5

Labyrinthe 6×9; 7 obstacles; 54 cases × 4 actions. Oracle indépendant BFS : 14 pas, retour optimal -13.


## 2. Fabriquer les datasets : trois politiques de comportement

En RL offline, le dataset est une donnee du problème — mais pour etudier le phenomene en laboratoire, nous le fabriquons nous-mêmes. Nous entrainons d'abord **online** (Q-learning classique du notebook 5) deux politiques de qualites différentes, puis nous enregistrons leurs trajectoires comme de simples logs :

| Dataset | Politique de comportement | Analogue reel |
|---------|---------------------------|---------------|
| **expert** | Q-learning converge (300 episodes), $\epsilon = 0.1$ | demonstrations d'un opérateur competent |
| **medium** | Q-learning interrompu (20 episodes), $\epsilon = 0.3$ | logs d'un ancien controleur mediocre |
| **random** | actions uniformes ($\epsilon = 1$) | exploration aveugle, données de capteurs bruts |

Pour chaque dataset nous mesurons deux choses : la **couverture** (fraction des paires $(s, a)$ presentes au moins une fois) et le **retour moyen** de la politique de comportement. Ces deux axes — qualite et couverture — sont orthogonaux, et chaque méthode offline echoue sur un axe différent.

In [3]:
record Transition(int S,int A,double R,int S2,bool Done);
int EpsilonGreedy(double[,] q,int s,double eps,Random rng){if(rng.NextDouble()<eps)return rng.Next(4);double max=Enumerable.Range(0,4).Max(a=>q[s,a]);var ties=Enumerable.Range(0,4).Where(a=>q[s,a]==max).ToArray();return ties[rng.Next(ties.Length)];}
double[,] QLearningOnline(Maze e,int episodes,int seed){var rng=new Random(seed);var q=new double[Maze.NStates,4];for(int ep=0;ep<episodes;ep++){int s=e.Id(e.Start);for(int t=0;t<MAX_STEPS;t++){int a=EpsilonGreedy(q,s,EPSILON,rng);var (s2,r,done)=e.Step(s,a);double target=done?r:r+GAMMA*Enumerable.Range(0,4).Max(x=>q[s2,x]);q[s,a]+=ALPHA*(target-q[s,a]);if(done)break;s=s2;}}return q;}
(int action,double meanReturn,double success)[] Evaluate(Maze e,int[] policy,int episodes=20,int seed=123){var rng=new Random(seed);var rows=new List<(int,double,bool)>();for(int ep=0;ep<episodes;ep++){int s=e.Id(e.Start);double ret=0;bool ok=false;for(int t=0;t<MAX_STEPS;t++){int a=policy[s];if(a<0)a=rng.Next(4);var z=e.Step(s,a);s=z.state;ret+=z.reward;if(z.done){ok=true;break;}}rows.Add((ep,ret,ok));}return rows.Select(_=>(0,rows.Average(x=>x.Item2),rows.Count(x=>x.Item3)/(double)episodes)).ToArray();}
int[] Greedy(double[,] q)=>Enumerable.Range(0,q.GetLength(0)).Select(s=>Enumerable.Range(0,4).OrderByDescending(a=>q[s,a]).First()).ToArray();
(List<Transition> data,double meanReturn) Collect(Maze e,double[,] behavior,double eps,int episodes,int seed){var rng=new Random(seed);var d=new List<Transition>();double sum=0;for(int ep=0;ep<episodes;ep++){int s=e.Id(e.Start);double ret=0;for(int t=0;t<MAX_STEPS;t++){int a=EpsilonGreedy(behavior,s,eps,rng);var (s2,r,done)=e.Step(s,a);d.Add(new Transition(s,a,r,s2,done));ret+=r;if(done)break;s=s2;}sum+=ret;}return(d,sum/episodes);}
double Coverage(Maze e,List<Transition> d)=>d.Select(x=>(x.S,x.A)).Distinct().Count()/(double)(Maze.NStates*4);
Console.WriteLine("Algorithmes définis : Q-learning online, politique epsilon-greedy avec départage aléatoire des ex æquo, évaluation 20 épisodes et collecte hors-ligne.");

Algorithmes définis : Q-learning online, politique epsilon-greedy avec départage aléatoire des ex æquo, évaluation 20 épisodes et collecte hors-ligne.


**Lecture.** Les méthodes de collecte sont définies; la cellule suivante entraîne les comportements et mesure les trois jeux de données.

In [4]:
var Qexpert=QLearningOnline(env,300,SEED);var Qmedium=QLearningOnline(env,20,SEED);
var retExpert=Evaluate(env,Greedy(Qexpert))[0].meanReturn;var retMedium=Evaluate(env,Greedy(Qmedium))[0].meanReturn;
var datasets=new Dictionary<string,List<Transition>>();var behaviorReturns=new Dictionary<string,double>();
foreach(var item in new[]{("expert",Qexpert,0.1,1),("medium",Qmedium,0.3,2),("random",Qexpert,1.0,3)}){var got=Collect(env,item.Item2,item.Item3,50,item.Item4);datasets[item.Item1]=got.data;behaviorReturns[item.Item1]=got.meanReturn;}
Console.WriteLine($"Expert online 300 épisodes : {retExpert:F1}; medium 20 épisodes : {retMedium:F1}");
Console.WriteLine($"{"dataset",-8} {"transitions",11} {"couverture",11} {"retour comportement",20}");
foreach(var (name,d) in datasets)Console.WriteLine($"{name,-8} {d.Count,11} {Coverage(env,d),10:P0} {behaviorReturns[name],20:F1}");

Expert online 300 épisodes : -13.0; medium 20 épisodes : -200.0


dataset  transitions  couverture  retour comportement


expert          1126       39 %                -21.5


medium          9141       78 %               -182.6


random          9510       85 %               -190.1


**Lecture des données mesurées.** Les trois jeux séparent qualité et couverture : expert 39 % et retour comportement -21,5; medium 78 % et -182,6; random 85 % et -190,1. La couverture augmente avec l’exploration, tandis que les retours se dégradent. Les nombres diffèrent du notebook Python car le générateur pseudo-aléatoire .NET n’est pas NumPy.

## 3. Behavior Cloning : imiter le dataset

La méthode la plus simple ignore completement les recompenses : dans chaque etat, jouer l'**action majoritaire** du dataset. C'est la version tabulaire du *Behavior Cloning* — en deep, on entrainerait un classifieur $s \mapsto a$ par maximum de vraisemblance sur les paires $(s_i, a_i)$.

$$\pi_{BC}(s) = \arg\max_a \; \#\{i : s_i = s,\; a_i = a\}$$

Dans un etat jamais visite par le dataset, la politique n'a aucune information : nous tirons une action au hasard (c'est le *distribution shift* de l'imitation learning — des qu'on quitte les etats du dataset, on improvise).

In [5]:
int[] BcPolicy(Maze e,List<Transition> data){var counts=new int[Maze.NStates,4];foreach(var x in data)counts[x.S,x.A]++;var p=Enumerable.Repeat(-1,Maze.NStates).ToArray();for(int s=0;s<Maze.NStates;s++){if(Enumerable.Range(0,4).Sum(a=>counts[s,a])==0)continue;int max=Enumerable.Range(0,4).Max(a=>counts[s,a]);p[s]=Enumerable.Range(0,4).First(a=>counts[s,a]==max);}return p;}
(double[,] q,int[] policy,HashSet<int>[] seen) OfflineQ(Maze e,List<Transition> data,int passes=300,bool constrain=false,int seed=0,double alpha=0.1){var rng=new Random(seed);var values=new double[Maze.NStates,4];var seen=Enumerable.Range(0,Maze.NStates).Select(_=>new HashSet<int>()).ToArray();foreach(var x in data)seen[x.S].Add(x.A);for(int pass=0;pass<passes;pass++){foreach(int i in Enumerable.Range(0,data.Count).OrderBy(_=>rng.Next())){var x=data[i];double target=x.R;if(!x.Done){int[] acts=constrain?seen[x.S2].ToArray():Enumerable.Range(0,4).ToArray();double mx=acts.Length==0?0:acts.Max(a=>values[x.S2,a]);target+=GAMMA*mx;}values[x.S,x.A]+=alpha*(target-values[x.S,x.A]);}}var policy=Enumerable.Repeat(-1,Maze.NStates).ToArray();for(int s=0;s<Maze.NStates;s++){if(seen[s].Count==0)continue;int[] acts=constrain?seen[s].ToArray():Enumerable.Range(0,4).ToArray();if(acts.Length>0)policy[s]=acts.OrderByDescending(a=>values[s,a]).First();}return(values,policy,seen);}


var resultsBc=new Dictionary<string,double>();Console.WriteLine($"{"dataset",-8} {"retour BC",10} {"succès",8} {"comportement",16}");
foreach(var (name,d) in datasets){var ev=Evaluate(env,BcPolicy(env,d))[0];resultsBc[name]=ev.meanReturn;Console.WriteLine($"{name,-8} {ev.meanReturn,10:F1} {ev.success,7:P0} {behaviorReturns[name],16:F1}");}
Console.WriteLine("BC choisit la première action en cas d’égalité des comptes; un état sans données garde la sentinelle -1 et sera évalué par action aléatoire.");

dataset   retour BC   succès     comportement


expert        -13.0   100 %            -21.5


medium       -200.0     0 %           -182.6


random       -200.0     0 %           -190.1


BC choisit la première action en cas d’égalité des comptes; un état sans données garde la sentinelle -1 et sera évalué par action aléatoire.


**Lecture de BC.** Sur ce run, BC obtient -13,0 sur l’expert, contre -21,5 pour le comportement : l’action majoritaire filtre une partie du bruit d’exploration. Sur medium et random, BC obtient -200,0 et 0 % de succès. BC n’utilise pas les récompenses; sa qualité dépend des états/actions couverts et de la façon dont les démonstrations déterminent les majorités. Il n’est donc pas en général plafonné au retour moyen de la politique de comportement.

<a id="exercice-taille"></a>

### Exercice 1 — Combien de demonstrations faut-il ?

La qualite des méthodes offline depend de la **taille** du dataset, pas seulement de sa qualite. Collectez des datasets experts de tailles croissantes et comparez BC et BCQ-lite : lequel est le plus econome en données ? A partir de combien d'episodes chacun devient-il fiable ?

In [6]:
// TODO étudiant : réaliser l’ablation des tailles [1, 2, 5, 10, 50].
int[] tailles={1,2,5,10,50};
double[] retoursBcEx1=null; // TODO étudiant : une mesure par taille, valeurs finies
 double[] retoursBcqEx1=null; // TODO étudiant : une mesure par taille, valeurs finies
if(retoursBcEx1 is null || retoursBcqEx1 is null) Console.WriteLine("Exercice 1 à compléter : comparer BC et BCQ-lite selon la taille.");
else { double bcqExpertBaseline=Evaluate(env,OfflineQ(env,datasets["expert"],constrain:true).policy)[0].meanReturn; if(retoursBcEx1.Length!=tailles.Length || retoursBcqEx1.Length!=tailles.Length || retoursBcEx1.Concat(retoursBcqEx1).Any(x=>!double.IsFinite(x) || x < -MAX_STEPS || x > -13) || Math.Abs(retoursBcEx1[^1]-resultsBc["expert"])>1e-9 || Math.Abs(retoursBcqEx1[^1]-bcqExpertBaseline)>1e-9) Console.WriteLine("Résultats invalides : cinq mesures finies dans [-200,-13], avec le point taille 50 égal aux baselines expert mesurées.");
else Console.WriteLine($"Résultats reçus : BC={string.Join(",",retoursBcEx1)}; BCQ={string.Join(",",retoursBcqEx1)}"); }

Exercice 1 à compléter : comparer BC et BCQ-lite selon la taille.


## 4. Q-learning offline naif : l'erreur d'extrapolation

Le Q-learning est *off-policy* : sa cible $r + \gamma \max_{a'} Q(s', a')$ ne depend pas de la politique qui a produit la transition. Il est donc tentant de le faire tourner directement sur le dataset, en rejouant les transitions en boucle (comme un replay buffer fige, sans jamais y ajouter de nouvelles expériences) :

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\big[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\big] \qquad (s, a, r, s') \sim \mathcal{D}$$

Le piege est dans le $\max_{a'}$ : il porte sur **toutes** les actions, y compris celles **jamais observees en $s'$**. Pour ces actions fantomes, $Q(s', a')$ garde sa valeur d'initialisation $0$ — qui, avec notre recompense de $-1$ par pas, est **optimiste** (la vraie valeur est negative partout sauf au but). Le bootstrap recopie alors cet optimisme : chaque cible vaut $-1 + \gamma \cdot 0 \approx -1$, toutes les valeurs s'ecrasent vers $-1$, et la politique greedy prefere systematiquement les actions inconnues aux actions documentees. En ligne, l'agent essaierait ces actions, serait decu, et corrigerait. **Offline, rien ne vient jamais corriger l'illusion.**

In [7]:
var resultsNaive=new Dictionary<string,double>();Console.WriteLine($"{"dataset",-8} {"retour Q naïf",14} {"succès",8} {"BC",8}");
foreach(var (name,d) in datasets){var (_,pol,_)=OfflineQ(env,d);var ev=Evaluate(env,pol)[0];resultsNaive[name]=ev.meanReturn;Console.WriteLine($"{name,-8} {ev.meanReturn,14:F1} {ev.success,7:P0} {resultsBc[name],8:F1}");}
Console.WriteLine("Q-learning hors-ligne rejoue 300 passes sur les transitions mélangées; la cible naïve maximise les 4 actions, y compris hors support.");

dataset   retour Q naïf   succès       BC


expert           -200.0     0 %    -13.0


medium           -200.0     0 %   -200.0


random            -13.0   100 %   -200.0


Q-learning hors-ligne rejoue 300 passes sur les transitions mélangées; la cible naïve maximise les 4 actions, y compris hors support.


**Lecture du Q-learning naïf.** Les retours observés sont -200,0 sur expert et medium, et -13,0 sur random. Le jeu random couvre 85 % des 216 paires état-action; les lacunes y sont moins susceptibles d’offrir des actions fantômes au maximum. La couverture seule ne garantit pas le succès, comme le montrent ces résultats particuliers; c’est le mécanisme d’extrapolation qui est étudié ici.

### Fixtures indépendantes

Ces contrôles exercent réellement `OfflineQ` sur de petits datasets déterministes. Ils vérifient la cible terminale, les politiques d’états non couverts et la différence entre un bootstrap naïf et un bootstrap restreint au support.

In [8]:
// Fixtures indépendantes : datasets vides, état inconnu, terminaison et bootstrap naïf/contraint.
var terminalFixture=OfflineQ(env,new List<Transition>{new(0,0,-1,1,true)},passes:1,alpha:0.1);
var emptyNaiveFixture=OfflineQ(env,new List<Transition>(),passes:1,constrain:false);
var emptyConstrainedFixture=OfflineQ(env,new List<Transition>(),passes:1,constrain:true);
var oneStateData=new List<Transition>{new(0,0,-1,1,true)};
var unknownNaiveFixture=OfflineQ(env,oneStateData,passes:1,constrain:false);
var unknownConstrainedFixture=OfflineQ(env,oneStateData,passes:1,constrain:true);
var supportFixture=new List<Transition>{new(0,0,-1,1,false),new(1,2,-1,2,true)};
var naiveSupportFixture=OfflineQ(env,supportFixture,passes:2,constrain:false,seed:0,alpha:1.0);
var constrainedSupportFixture=OfflineQ(env,supportFixture,passes:2,constrain:true,seed:0,alpha:1.0);
var emptySuccessorFixture=OfflineQ(env,new List<Transition>{new(0,0,-1,1,false)},passes:1,constrain:true,seed:0,alpha:1.0);
var bcFixture=BcPolicy(env,new List<Transition>{new(0,0,-1,1,false),new(0,1,-1,1,false)});
var tieFixture=EpsilonGreedy(new double[Maze.NStates,4],0,0.0,new Random(9));
double pairCoverage=Coverage(env,new List<Transition>{new(0,0,-1,1,false)});
var originalCulture=CultureInfo.CurrentCulture;CultureInfo.CurrentCulture=new CultureInfo("fr-FR");
string cultureSvg=$"<svg width='{SvgNumber(1.25)}' height='{SvgNumber(-2.5)}'/>";
var cultureRoot=XDocument.Parse(cultureSvg).Root;CultureInfo.CurrentCulture=originalCulture;
if(Math.Abs(terminalFixture.q[0,0]+0.1)>1e-12 || emptyNaiveFixture.policy.Any(a=>a!=-1) || emptyConstrainedFixture.policy.Any(a=>a!=-1) || unknownConstrainedFixture.policy[0]!=0 || unknownNaiveFixture.policy.Skip(1).Any(a=>a!=-1) || unknownConstrainedFixture.policy.Skip(1).Any(a=>a!=-1) || Math.Abs(emptySuccessorFixture.q[0,0]+1.0)>1e-12 || Math.Abs(naiveSupportFixture.q[0,0]+1.0)>1e-12 || Math.Abs(constrainedSupportFixture.q[0,0]+1.95)>1e-12 || bcFixture[0]!=0 || bcFixture[1]!=-1 || tieFixture<0 || tieFixture>=4 || Math.Abs(pairCoverage-1.0/216)>1e-12 || cultureRoot?.Attribute("width")?.Value!="1.25" || cultureRoot?.Attribute("height")?.Value!="-2.5") throw new Exception("Fixture indépendant RL offline en échec");
Console.WriteLine($"Fixtures OfflineQ OK : terminal={terminalFixture.q[0,0]:F2}; dataset vide - politiques -1/54 états; états inconnus hors dataset -1; action finale BCQ connue=0; support vide={emptySuccessorFixture.q[0,0]:F2}; cibles OfflineQ naïf={naiveSupportFixture.q[0,0]:F2}, contraint={constrainedSupportFixture.q[0,0]:F2}; BC tie/inconnu={bcFixture[0]}/{bcFixture[1]}; epsilon tie={tieFixture}; couverture={pairCoverage:F6}; SVG fr-FR={cultureRoot?.Attribute("width")?.Value}/{cultureRoot?.Attribute("height")?.Value}.");


Fixtures OfflineQ OK : terminal=-0.10; dataset vide - politiques -1/54 états; états inconnus hors dataset -1; action finale BCQ connue=0; support vide=-1.00; cibles OfflineQ naïf=-1.00, contraint=-1.95; BC tie/inconnu=0/-1; epsilon tie=1; couverture=0.004630; SVG fr-FR=1.25/-2.5.


**Lecture des fixtures.** Les contrôles confirment que les états sans transitions gardent la sentinelle -1 dans les deux variantes, et que les deux appels `OfflineQ` diverent sur l’action non documentée du successeur.

In [9]:
var (Qnaive,_,seenExp)=OfflineQ(env,datasets["expert"]);var visited=Enumerable.Range(0,Maze.NStates).Where(s=>seenExp[s].Count>0).ToArray();int unseenGreedy=visited.Count(s=>!seenExp[s].Contains(Enumerable.Range(0,4).OrderByDescending(a=>Qnaive[s,a]).First()));
Console.WriteLine($"États visités par les données expertes : {visited.Length}");Console.WriteLine($"Choix greedy hors support : {unseenGreedy}/{visited.Length} ({unseenGreedy/(double)visited.Length:P0})");
int zoom=env.Id((3,4));var qvals=Enumerable.Range(0,4).Select(a=>Qnaive[zoom,a]).ToArray();string bars="<svg xmlns='http://www.w3.org/2000/svg' width='560' height='290' viewBox='0 0 560 290'><rect width='100%' height='100%' fill='white'/><text x='20' y='24' font-size='16'>Q naïf en (3, 4) — bleu : vu; rouge : hors support</text><line x1='60' y1='50' x2='530' y2='50' stroke='black'/><text x='20' y='150' transform='rotate(-90 20 150)' font-size='12'>Q</text><text x='35' y='54' font-size='11'>0</text><text x='35' y='135' font-size='11'>-1</text><text x='35' y='215' font-size='11'>-2</text>";string[] an={"haut","bas","gauche","droite"};for(int a=0;a<4;a++){double h=Math.Max(2,Math.Abs(qvals[a])*80);bars+=$"<rect x='{85+a*110}' y='50' width='58' height='{SvgNumber(h)}' fill='{(seenExp[zoom].Contains(a)?"#2878b5":"#cf4b45")}'/><text x='{114+a*110}' y='252' text-anchor='middle'>{an[a]}</text><text x='{114+a*110}' y='{SvgNumber(68+h)}' text-anchor='middle'>{qvals[a].ToString("F2",invariant)}</text>";}bars+="</svg>";record DiagnosticSvg(string Content);
Formatter.Register<DiagnosticSvg>((v,w) => w.Write(v.Content), "text/html");
display(new DiagnosticSvg(bars));Console.WriteLine($"Q en (3,4) : [{string.Join(", ",qvals.Select(x=>x.ToString("F2",invariant)))}]; actions observées : {string.Join(", ",seenExp[zoom].Order().Select(a=>an[a]))}");

États visités par les données expertes : 40


Choix greedy hors support : 30/40 (75 %)


Q naïf en (3, 4) — bleu : vu; rouge : hors support Q 0 -1 -2 haut 0.00 bas 0.00 gauche 0.00 droite -1.95

Q en (3,4) : [0.00, 0.00, 0.00, -1.95]; actions observées : droite


Dans ce run .NET, les données expertes visitent 40 états; la politique greedy choisit une action hors support dans 30 d’entre eux (75 %). Au zoom (3,4), seule l’action « droite » est présente; les valeurs mesurées sont [0,00, 0,00, 0,00, -1,95]. Les trois actions hors support restent à zéro et le Q-learning naïf en choisit une par départage des ex æquo. Le détail des valeurs et du support est affiché juste au-dessus dans la figure.

<a id="exercice-penalite"></a>

### Exercice 2 — Penalite conservatrice (CQL-lite tabulaire)

CQL remplace l'interdiction stricte de BCQ par une **penalite** : les actions hors dataset gardent le droit d'exister, mais leur valeur est artificiellement abaissee. En tabulaire, la version la plus simple s'applique *après* l'apprentissage naif :

$$Q_{pen}(s, a) = Q_{naif}(s, a) - \beta \cdot \mathbb{1}\big[(s, a) \notin \mathcal{D}\big]$$

Evaluez la politique greedy sur $Q_{pen}$ pour plusieurs $\beta$ sur le dataset **medium**. Que retrouve-t-on en $\beta = 0$ ? Vers quoi tend le comportement quand $\beta$ grandit ?

In [10]:
// TODO étudiant : écrire le helper, puis entraîner le Q offline naïf et appliquer le masque de support.
double[,] PenaliserHorsSupport(double[,] q,bool[,] vu,double beta){return null;} // TODO étudiant : copier q et soustraire beta hors support
double[] betas={0,0.5,2,10};
double[] retoursCql=null; // TODO étudiant : une mesure par beta sur datasets["medium"]
var qFixture=new double[,]{{1.5,-0.2},{0.3,2.0}};var vuFixture=new bool[,]{{true,false},{false,true}};
var fixtureBeta0=PenaliserHorsSupport(qFixture,vuFixture,0.0);var fixtureBeta2=PenaliserHorsSupport(qFixture,vuFixture,2.0);
bool fixtureValide=fixtureBeta0!=null&&fixtureBeta2!=null&&fixtureBeta0.GetLength(0)==2&&fixtureBeta0.GetLength(1)==2&&fixtureBeta2.GetLength(0)==2&&fixtureBeta2.GetLength(1)==2;
if(fixtureValide){fixtureValide=Math.Abs(fixtureBeta0[0,0]-1.5)<1e-12&&Math.Abs(fixtureBeta0[0,1]+0.2)<1e-12&&Math.Abs(fixtureBeta0[1,0]-0.3)<1e-12&&Math.Abs(fixtureBeta0[1,1]-2.0)<1e-12&&Math.Abs(fixtureBeta2[0,0]-1.5)<1e-12&&Math.Abs(fixtureBeta2[0,1]-(-2.2))<1e-12&&Math.Abs(fixtureBeta2[1,0]-(-1.7))<1e-12&&Math.Abs(fixtureBeta2[1,1]-2.0)<1e-12;}
if(retoursCql is null) Console.WriteLine("Exercice 2 à compléter : implémenter PenaliserHorsSupport, vérifier la fixture beta 0/2, puis mesurer sur medium.");
else if(retoursCql.Length!=betas.Length || retoursCql.Any(x=>!double.IsFinite(x)||x < -MAX_STEPS||x > -13) || Math.Abs(retoursCql[0]-resultsNaive["medium"])>1e-9 || !fixtureValide) Console.WriteLine("Résultats invalides : quatre mesures finies dans [-200,-13], beta=0 égal au Q naïf medium, et helper conforme à la fixture non triviale.");
else Console.WriteLine($"Mesures CQL-lite : {string.Join(",",retoursCql)}; fixture helper non triviale OK.");

Exercice 2 à compléter : implémenter PenaliserHorsSupport, vérifier la fixture beta 0/2, puis mesurer sur medium.


## 5. Contraindre au support du dataset : BCQ-lite

Le diagnostic dicte le remede : interdire au $\max$ de regarder les actions fantomes. On restreint le bootstrap **au support du dataset** :

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\Big[r + \gamma \max_{a' \,\in\, \mathcal{A}_\mathcal{D}(s')} Q(s', a') - Q(s, a)\Big]$$

ou $\mathcal{A}_\mathcal{D}(s') = \{a' : (s', a') \in \mathcal{D}\}$ est l'ensemble des actions effectivement observees en $s'$. La politique finale est elle aussi restreinte aux actions observees. C'est la version tabulaire de **BCQ** (*Batch-Constrained deep Q-learning*, Fujimoto et al., 2019) — en continu, un modèle generatif estime $\pi_\beta(a|s)$ et le $\max$ ne porte que sur des actions plausibles sous le comportement.

L'enjeu : garder la capacite du Q-learning a **recombiner** des bouts de trajectoires (ce que BC ne sait pas faire), sans son optimisme hors support.

In [11]:
var resultsBcq=new Dictionary<string,double>();Console.WriteLine($"{"dataset",-8} {"retour BCQ-lite",16} {"succès",8} {"comportement",16}");
foreach(var (name,d) in datasets){var (_,pol,_)=OfflineQ(env,d,constrain:true);var ev=Evaluate(env,pol)[0];resultsBcq[name]=ev.meanReturn;Console.WriteLine($"{name,-8} {ev.meanReturn,16:F1} {ev.success,7:P0} {behaviorReturns[name],16:F1}");}
Console.WriteLine("Le bootstrap et la politique finale sont restreints aux actions observées; un état successeur sans support utilise une valeur bootstrap nulle.");

dataset   retour BCQ-lite   succès     comportement


expert              -13.0   100 %            -21.5


medium              -13.0   100 %           -182.6


random              -13.0   100 %           -190.1


Le bootstrap et la politique finale sont restreints aux actions observées; un état successeur sans support utilise une valeur bootstrap nulle.


**Lecture.** BCQ-lite a été mesuré sur les trois datasets; le graphique suivant rassemble BC, Q naïf et BCQ-lite avec les retours comportementaux.

<a id="exercice-passes"></a>

### Exercice 3 — Convergence : combien de passes sur le dataset ?

Un detail d'implementation qui n'en est pas un : `OfflineQ` rejoue le dataset `n_passes = 300` fois. Mesurez les performances à chaque budget et expliquez l’effet des passes sur la valeur de l’action gauche, qui heurte le mur au départ. Le résultat observé peut dépendre de l’ordre pseudo-aléatoire des transitions.

In [12]:
// TODO étudiant : comparer les passes [10, 50, 100, 300] sur le dataset expert.
int[] listePasses={10,50,100,300};
double[] retoursPasses=null; // TODO étudiant : une mesure finie par budget
if(retoursPasses is null) Console.WriteLine("Exercice 3 à compléter : mesurer l’effet du nombre de passes.");
else if(retoursPasses.Length!=listePasses.Length || retoursPasses.Any(x=>!double.IsFinite(x) || x < -MAX_STEPS || x > -13) || Math.Abs(retoursPasses[^1]-resultsBcq["expert"])>1e-9) Console.WriteLine("Résultats invalides : quatre mesures finies dans [-200,-13], avec le point passes=300 égal à la mesure BCQ expert.");
else Console.WriteLine($"Retours selon passes : {string.Join(",",retoursPasses)}");

Exercice 3 à compléter : mesurer l’effet du nombre de passes.


**Lecture du protocole.** L’ablation du nombre de passes reste à mesurer par l’étudiant. Le graphique suivant regroupe les résultats déjà mesurés dans le notebook.

In [13]:
var chartCulture=CultureInfo.CurrentCulture;CultureInfo.CurrentCulture=new CultureInfo("fr-FR");var methods=new[]{("Behavior Cloning",resultsBc,"#4477aa"),("Q offline naïf",resultsNaive,"#ee6677"),("BCQ-lite",resultsBcq,"#228833")};string[] ds={"expert","medium","random"};string chart="<svg xmlns='http://www.w3.org/2000/svg' width='760' height='390' viewBox='0 0 760 390'><rect width='100%' height='100%' fill='white'/><text x='24' y='24' font-size='17'>Retours moyens (20 épisodes, optimum -13)</text><line x1='70' y1='310' x2='720' y2='310' stroke='black'/><line x1='70' y1='60' x2='70' y2='310' stroke='black'/><text x='22' y='190' transform='rotate(-90 22 190)' font-size='12'>Retour moyen (points)</text>";for(int t=0;t<=4;t++){int y=60+t*62;int val=0-t*50;chart+=$"<line x1='66' y1='{y}' x2='720' y2='{y}' stroke='#ddd'/><text x='58' y='{y+4}' text-anchor='end' font-size='12'>{val}</text>";}for(int i=0;i<3;i++){int center=165+i*210;chart+=$"<text x='{center}' y='337' text-anchor='middle'>{ds[i]}</text>";for(int j=0;j<3;j++){double val=methods[j].Item2[ds[i]];double height=(val+200)*1.24;chart+=$"<rect x='{center-47+j*34}' y='{SvgNumber(310-height)}' width='28' height='{SvgNumber(height)}' fill='{methods[j].Item3}'/>";if(Math.Abs(val+200)<1e-9)chart+=$"<text x='{center-33+j*34}' y='304' text-anchor='middle' font-size='10'>-200</text>";}double behaviorY=310-(behaviorReturns[ds[i]]+200)*1.24;chart+=$"<polygon points='{center},{SvgNumber(behaviorY-6)} {center+6},{SvgNumber(behaviorY)} {center},{SvgNumber(behaviorY+6)} {center-6},{SvgNumber(behaviorY)}' fill='#222'/>";}for(int j=0;j<3;j++){chart+=$"<rect x='{85+j*150}' y='360' width='14' height='14' fill='{methods[j].Item3}'/><text x='{104+j*150}' y='372' font-size='12'>{methods[j].Item1}</text>";}chart+="<line x1='70' y1='76' x2='720' y2='76' stroke='#228833' stroke-dasharray='5,4'/><polygon points='545,351 551,357 545,363 539,357' fill='#222'/><text x='559' y='363' font-size='12'>Comportement</text></svg>";var chartXml=XDocument.Parse(chart);if(chartXml.Root?.Name.LocalName!="svg"||chart.Contains("1,24"))throw new Exception("Attribut SVG non invariant sous fr-FR");CultureInfo.CurrentCulture=chartCulture;record ComparativeSvg(string Content);
Formatter.Register<ComparativeSvg>((v,w) => w.Write(v.Content), "text/html");
display(new ComparativeSvg(chart));
Console.WriteLine("Comparaison affichée : les trois méthodes offline et la référence optimale -13; retours effectivement mesurés dans les cellules précédentes.");

Retours moyens (20 épisodes, optimum -13) Retour moyen (points) 0 -50 -100 -150 -200 expert -200 medium -200 -200 random -200 Behavior Cloning Q offline naïf BCQ-lite Comportement

Comparaison affichée : les trois méthodes offline et la référence optimale -13; retours effectivement mesurés dans les cellules précédentes.


**Lecture des résultats mesurés.** BC atteint -13,0 sur expert puis -200,0 sur medium et random. Le Q naïf obtient -200,0, -200,0 et -13,0. BCQ-lite atteint -13,0 sur les trois jeux. Sur le jeu medium, le dataset contient des transitions qui permettent de reconstruire un chemin court malgré le retour moyen comportemental -182,6 : c’est le stitching observé dans cette exécution.

## 6. Du tabulaire au deep : BCQ, CQL et le pont vers RLHF/DPO

En dimension reelle (etats continus, $Q$ approxime par un reseau), « l'ensemble des actions vues en $s'$ » n'existe plus litteralement — chaque etat est unique. Les méthodes modernes traduisent la même idee de trois facons :

- **BCQ** (Fujimoto et al., 2019) : un auto-encodeur variationnel modelise $\pi_\beta(a|s)$ ; le $\max$ ne porte que sur des actions echantillonnees de ce modèle (contrainte de support *explicite*).
- **CQL** (*Conservative Q-Learning*, Kumar et al., 2020) : plutot que d'interdire, **penaliser** — un terme de regularisation pousse $Q$ vers le bas sur les actions hors distribution et vers le haut sur les actions du dataset. Le $Q$ appris est un *minorant* du vrai $Q$ : l'optimisme hors support devient du pessimisme (l'exercice 2 en construit la version tabulaire).
- **IQL, TD3+BC, Decision Transformer...** : la famille est riche, mais le principe est invariant — contrôler l'ecart entre la politique apprise et la politique de comportement.

**Le pont vers les LLM.** L'alignement d'un modèle de langage est un problème de RL offline deguise. Le *fine-tuning supervise* (SFT) est exactement du Behavior Cloning sur des demonstrations humaines. **RLHF** optimise ensuite une recompense apprise, avec une penalite KL qui force la politique a rester proche du modèle de reference — le même rôle que la contrainte de support de BCQ : sans elle, la politique exploite les zones ou le modèle de recompense extrapole mal (*reward hacking*, le jumeau de notre erreur d'extrapolation). **DPO** (Rafailov et al., 2023) pousse la logique au bout : il elimine le RL et reecrit l'objectif RLHF comme une simple classification de préférences sous contrainte KL — de l'offline pur, sans interaction. La boucle est bouclee : les idees nees sur des labyrinthes 6x9 gouvernent aujourd'hui l'entrainement des plus grands modèles.

## 7. Exercices — protocoles à compléter

Les exercices sont répartis dans le parcours : [taille des démonstrations](#exercice-taille) après BC, [pénalité conservatrice CQL-lite](#exercice-penalite) après le diagnostic du Q offline, et [nombre de passes](#exercice-passes) après BCQ-lite. Les cellules restent volontairement incomplètes et s’exécutent sans erreur.

Les gardes contrôlent les dimensions, la finitude et la plage de retours [-200,-13]. Les ancres comparent la taille 50 aux baselines expert mesurées, beta=0 au Q naïf medium, et passes=300 au BCQ expert. Une fixture matricielle vérifie le helper de pénalisation sur des actions observées et hors support. Ces contrôles ne certifient pas des courbes arbitraires comme si elles avaient été exécutées.

Les trois exercices reutilisent les fonctions du notebook (`Collect`, `BcPolicy`, `OfflineQ`, `Evaluate`). Completez les cellules — elles doivent s'executer sans erreur même incompletes.

## À retenir

Ce notebook a parcouru le RL offline de bout en bout sur un labyrinthe contrôlé :

- Le **Behavior Cloning** imite les actions majoritaires sans exploiter les récompenses. Dans cette exécution il améliore le retour expert de -21,5 à -13,0, mais échoue sur medium et random. Le retour obtenu dépend des données et du départage des actions majoritaires.
- Le **Q-learning naïf sur dataset figé** peut échouer même sur des données expertes : son maximum bootstrappe sur toutes les actions, y compris celles jamais observées, et aucune interaction ne vient corriger les valeurs extrapolées. Il réussit ici sur random, où la couverture mesurée est 85 %.
- La **contrainte de support** (BCQ-lite) limite le bootstrap aux actions observées et permet ici de recombiner les transitions : BCQ-lite obtient -13,0 sur expert, medium et random. C’est le stitching mesuré sur medium.
- Sur expert, 30 des 40 états visités ont une action greedy hors support (75 %). Au zoom (3,4), trois actions non observées restent à Q=0 et dépassent la valeur -1,95 de la seule action observée « droite ».
- Les résultats stochastiques varient selon le générateur pseudo-aléatoire; les principes reposent sur les supports et les cibles effectivement inspectés, pas sur une garantie de classement entre méthodes.

## Vérifiez votre compréhension

**1. Pourquoi le Q-learning naïf obtient-il des résultats différents sur expert et random ?**

<details><summary>Réponse</summary>

La couverture est de 39 % pour expert et 85 % pour random dans cette exécution. Le maximum sur les actions non observées peut conserver une valeur optimiste de zéro. Cette différence de support contribue à la différence observée, sans garantir le même classement sur d’autres tirages.

</details>

**2. Au zoom (3,4), pourquoi le Q naïf choisit-il une action absente du dataset ?**

<details><summary>Réponse</summary>

Seule « droite » est observée et vaut -1,95. Les autres actions gardent Q=0; le maximum préfère donc une action hors support.

</details>

**3. Que change la contrainte BCQ-lite ?**

<details><summary>Réponse</summary>

Elle restreint le maximum de bootstrap et le choix final aux actions présentes dans le dataset. Sur les trois jeux de cette exécution, BCQ-lite obtient -13,0 avec un taux de succès de 100 %.

</details>

## Pour aller plus loin

- **[RL-10 — Reward shaping et curriculum learning](RL-10-Reward-Shaping-Curriculum-Python.ipynb)** : le carnet suivant de la serie — de la contrainte de support a la contrainte de forme : le reward shaping reecrit la recompense, le curriculum reecrit l'ordre des donnees, et les deux se mesurent sur la meme question (combien d'episodes pour converger).
- **Notebook 5** : le Q-learning online dont tout part ; **notebook 6** : le replay buffer de DQN est déjà un mini-dataset offline (mais sans cesse rafraichi — c'est toute la différence) ;
- **Notebook 8** : Dyna-Q apprend un modèle des transitions ; le RL offline *model-based* (MOPO, MOReL) suit la même voie avec une penalite d'incertitude ;
- **[GenAI / FineTuning — FT-04 RLHF-DPO](../GenAI/FineTuning/FT-04-RLHF-DPO-Python.ipynb)** : SFT = Behavior Cloning, RLHF = RL contraint par KL, DPO = préférence learning offline — les notions de ce notebook, a l'echelle des LLM (pratique TRL/PEFT).
- **[GenAI / PostTraining](../GenAI/PostTraining/README.md)** : la chaîne complete SFT -> RLHF -> [DPO (PT-03)](../GenAI/PostTraining/PT_03_dpo_direct_preference.ipynb) -> GRPO -> RLVR, ou la contrainte de support de BCQ devient la penalite KL de DPO. C'est la ou atterrit le pont esquisse ici.

### Lectures

- Levine, Kumar, Tucker, Fu (2020), *Offline Reinforcement Learning: Tutorial, Review, and Perspectives on Open Problems*, arXiv:2005.01643
- Fujimoto, Meger, Precup (2019), *Off-Policy Deep Reinforcement Learning without Exploration* (BCQ), ICML
- Kumar, Zhou, Tucker, Levine (2020), *Conservative Q-Learning for Offline Reinforcement Learning* (CQL), NeurIPS
- Ross, Gordon, Bagnell (2011), *A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning* (DAgger), AISTATS
- Rafailov et al. (2023), *Direct Préférence Optimization: Your Language Model is Secretly a Reward Model*, NeurIPS
- Sutton & Barto (2018), *Reinforcement Learning: An Introduction*, chap. 8 (labyrinthe fig. 8.2)
